# 🦉🐳 El Agente de Excusas Docker — Sesión 12

**La situación (100% basada en hechos reales de hoy):** el profesor tiene que enseñar el servicio
corriendo en Docker... y Docker no arranca. Quedan minutos para la clase. Toca servirse del ingenio.

**La solución:** un agente con **2 tools** cuya misión es reunir **5 excusas certificadas** (credibilidad ≥ 7)
del catálogo oficial de 19 excusas, y redactar el comunicado épico para los alumnos.

Aunque la misión sea de risa, la anatomía es **exactamente** la del agente real de la S12 (`run_agent_s12.py`):

| Pieza | Aquí | En el agente real |
|---|---|---|
| **System prompt** | quién es el agente y cuándo parar | ídem |
| **User prompt** | "no me funciona el Docker, sálvame" | el transcript del cliente |
| **Tools** | `buscar_excusa`, `medir_credibilidad` | `search_budgets`, `derive_task_hours`... |
| **Bucle agéntico** | razona → actúa → observa, hasta 5 excusas buenas | ídem, hasta estimar todas las tareas |
| **Parada** | natural (el modelo deja de llamar tools) + `MAX_ITERATIONS` | ídem |
| **Traza** | `STEP N: reasoning / action / observation` | ídem |

> 💡 Fíjate en que **nadie programa el orden de las llamadas**: el modelo decide qué excusa consultar,
> cuáles medir, cuáles descartar por poco creíbles y cuándo parar. Eso es lo que lo hace *agente* y no *pipeline*.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
%pip -q install openai matplotlib

import getpass, json, random

try:  # En Colab: guarda OPENAI_API_KEY en el icono de la llave 🔑 (Secrets)
    from google.colab import userdata
    API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    API_KEY = getpass.getpass("OPENAI_API_KEY: ")

from openai import OpenAI
client = OpenAI(api_key=API_KEY)

MODEL = "gpt-5-mini"   # barato para el directo; sube a gpt-5 si quieres drama de calidad
MAX_ITERATIONS = 15    # el freno de mano de todo agente decente

## 1 · Las tools (el "mundo" que el agente puede tocar)

Dos tools, como manda el guion:

1. **`buscar_excusa(numero)`** — consulta el Catálogo Oficial de Excusas Docker™ (19 entradas, strings).
   El agente NO conoce el catálogo: tiene que explorarlo llamando a la tool. Igual que `search_budgets`
   no conoce el corpus hasta que busca.
2. **`medir_credibilidad(numero)`** — el Credibilómetro™: devuelve una nota **aleatoria** de 1 a 10.
   Es la observación *no determinista* del entorno — el agente tiene que adaptarse a lo que salga
   (si una excusa puntúa bajo, le toca buscar otra).

In [ ]:
# ── Implementación de las tools ────────────────────────────────────────────

CATALOGO_EXCUSAS = {
    1:  "El disco C: se ha quedado a 314 MB libres. No es que no quepa Docker: es que no cabe ni la excusa.",
    2:  "torch se ha traído a CUDA entera de acompañante: 7 GB de librerías NVIDIA para una GPU que no tengo.",
    3:  "El disco virtual de Docker ha engordado hasta 49 GB. Dice que son 'capas de invierno'.",
    4:  "WSL responde 'no connected VM', que en informático significa 'hoy no me apetece'.",
    5:  "El servicio com.docker.service estaba parado. Nadie sabe quién lo paró. Él jura que se paró solo.",
    6:  "La consola de Windows es cp1252 y las flechas → de mi traza le parecen arte moderno inaceptable.",
    7:  "La primera build tarda 40 minutos 'resolviendo dependencias', que es lo que dicen los procesos cuando no quieren dar explicaciones.",
    8:  "El puerto 3000 está ocupado por un proceso que juro solemnemente que no lancé yo.",
    9:  "Funciona en mi máquina... la del VPS. Esta es OTRA máquina. La cláusula letra pequeña del 'works on my machine'.",
    10: "Docker Desktop dice que necesita reiniciarse. Docker Desktop SIEMPRE necesita reiniciarse. Es su estado natural.",
    11: "Borré la caché de build para liberar espacio, y con ella, aparentemente, mi dignidad.",
    12: "El compose usa 'include:', que para mi versión de compose es literatura de ciencia ficción.",
    13: "La distro docker-desktop se desregistró y ahora Docker la busca como quien busca las llaves: en sitios donde ya miró tres veces.",
    14: "El corpus RAG necesita embeddings y OpenAI necesita mi tarjeta. Todo el mundo necesita algo en esta vida.",
    15: "Alembic dice que el esquema de la base de datos es 'del futuro'. Yo también querría ser del futuro y ver esta demo funcionando.",
    16: "El healthcheck lleva 20 minutos en 'starting'. Técnicamente no ha fallado: está madurando.",
    17: "El clasificador de permisos de mi IA me bloqueó el comando de arreglarlo. Hasta mi asistente desconfía de mí.",
    18: "Las capas de Docker se comparten entre imágenes... excepto, estadísticamente, cuando a mí me importa.",
    19: "En producción esto no pasa, dicen. Correcto: esto no es producción. Es peor. Es un DIRECTO.",
}

credibilometro = random.Random()   # sin semilla: cada directo es único, como los fallos de Docker
mediciones = []                    # (numero, nota) — para el plot final

def buscar_excusa(numero: int) -> dict:
    if numero not in CATALOGO_EXCUSAS:
        return {"error": f"El catálogo va del 1 al 19. El {numero} no existe (buena excusa, por cierto)."}
    return {"numero": numero, "excusa": CATALOGO_EXCUSAS[numero]}

def medir_credibilidad(numero: int) -> dict:
    if numero not in CATALOGO_EXCUSAS:
        return {"error": f"No puedo medir la credibilidad de una excusa que no existe ({numero})."}
    nota = credibilometro.randint(1, 10)
    mediciones.append((numero, nota))
    veredicto = "CERTIFICADA ✅" if nota >= 7 else "los alumnos no se lo tragan ❌"
    return {"numero": numero, "credibilidad": nota, "veredicto": veredicto}

def dispatch_tool(name: str, args: dict) -> dict:
    try:
        if name == "buscar_excusa":
            return buscar_excusa(int(args["numero"]))
        if name == "medir_credibilidad":
            return medir_credibilidad(int(args["numero"]))
        return {"error": f"tool desconocida: {name}"}
    except Exception as e:  # el error vuelve como observación: el agente se autocorrige
        return {"error": str(e)}

## 2 · Los schemas de las tools (lo que el MODELO ve)

El modelo no ve el código Python: ve **estos contratos**. Fíjate en las descripciones — son el
*prompt engineering* de las tools: le dicen al modelo cuándo y cómo usarlas. Planas y con
`strict: true`, como en la sesión.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "name": "buscar_excusa",
        "description": (
            "Consulta el Catálogo Oficial de Excusas Docker™ y devuelve el texto de la excusa. "
            "El catálogo tiene exactamente 19 excusas, numeradas del 1 al 19. "
            "No conoces su contenido de antemano: la única forma de saber qué dice una excusa es consultarla."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "numero": {"type": "integer", "description": "Número de excusa a consultar, de 1 a 19."}
            },
            "required": ["numero"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "medir_credibilidad",
        "description": (
            "Somete una excusa al Credibilómetro™, que devuelve una nota de 1 a 10. "
            "El aparato es caprichoso (aleatorio): la misma excusa puede puntuar distinto en otro momento. "
            "Una excusa se considera CERTIFICADA si saca 7 o más. Solo puedes medir excusas que ya hayas consultado."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "numero": {"type": "integer", "description": "Número (1-19) de la excusa a certificar."}
            },
            "required": ["numero"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

## 3 · System prompt y user prompt

- El **system prompt** define identidad, misión y — clave en agentes — **la condición de parada**.
- El **user prompt** es la petición desesperada del profesor. 100% verídica.

In [ ]:
SYSTEM_PROMPT = """Eres el Agente Oficial de Excusas de Julián, profesor del Máster en AI Engineering.
Situación: quedan minutos para la clase, había que enseñar el servicio corriendo en Docker, y Docker
ha decidido no arrancar. Tu misión es salvar el honor del profesor.

PROTOCOLO (síguelo con la seriedad de un ingeniero de guardia):
1. Explora el Catálogo Oficial de Excusas Docker™ (excusas 1 a 19) con `buscar_excusa`.
2. Somete las candidatas prometedoras al Credibilómetro™ con `medir_credibilidad`.
3. NO PARES hasta reunir exactamente 5 excusas CERTIFICADAS (credibilidad >= 7).
   Si una excusa puntúa por debajo de 7, queda descartada: busca y mide otras.
   El Credibilómetro es caprichoso; persevera, la ingeniería es así.
4. Cuando tengas las 5 certificadas, deja de llamar tools y redacta el COMUNICADO OFICIAL
   a los alumnos: tono épico-solemne de parte de guerra, pero cómico. Incluye las 5 excusas
   con su nota de credibilidad, y cierra prometiendo que la demo real llegará "en cuanto el
   contenedor supere sus problemas personales".

Sé eficiente: no consultes las 19 excusas si no hace falta, que la API la paga el profesor."""

USER_PROMPT = """No me funciona levantar el Docker y tengo que mostrar a los alumnos del máster
el servicio corriendo. No hay tiempo, no hay contenedores, solo me queda mi ingenio.
Necesito 5 excusas certificadas de máxima credibilidad, YA. Sálvame la clase."""

## 4 · El bucle agéntico (razona → actúa → observa)

El corazón de la sesión. Idéntico en estructura al del agente real:

- **Encadenado con estado**: `previous_response_id` — solo enviamos las observaciones nuevas,
  el servidor recuerda el resto de la conversación.
- **Parada natural**: el bucle termina cuando el modelo responde **sin** llamar a ninguna tool
  (ha cumplido el protocolo y escribe el comunicado).
- **`MAX_ITERATIONS`**: el freno de mano por si el Credibilómetro se pone cruel.
- **Los errores son observaciones**: si una tool falla, el error vuelve al modelo como resultado
  y este se autocorrige (pruébalo: pídele en el user prompt que consulte la excusa 42).

In [ ]:
trace = []          # [{step, reasoning, actions: [(tool, args, result)]}]
mediciones.clear()

def _reasoning_summary(resp):
    partes = []
    for item in resp.output:
        if item.type == "reasoning":
            for s in (item.summary or []):
                partes.append(s.text)
    return " ".join(partes) or "(el modelo no ha soltado prenda)"

resp = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_PROMPT,
    input=USER_PROMPT,
    tools=TOOLS,
    reasoning={"effort": "low", "summary": "auto"},
    store=True,
)

for step in range(1, MAX_ITERATIONS + 1):
    tool_calls = [item for item in resp.output if item.type == "function_call"]

    if not tool_calls:   # ← parada natural: el agente decidió que ha terminado
        print(f"⏹️  Parada natural en el paso {step}: el agente tiene sus 5 excusas y pasa a redactar.")
        break

    acciones, outputs = [], []
    for call in tool_calls:
        args = json.loads(call.arguments)
        result = dispatch_tool(call.name, args)
        acciones.append((call.name, args, result))
        outputs.append({
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(result, ensure_ascii=False),
        })

    trace.append({"step": step, "reasoning": _reasoning_summary(resp), "actions": acciones})
    print(f"STEP {step}: {len(acciones)} llamada(s) → " + ", ".join(f"{n}({a['numero']})" for n, a, _ in acciones))

    resp = client.responses.create(
        model=MODEL,
        previous_response_id=resp.id,   # ← el encadenado con estado
        input=outputs,                  # ← SOLO las observaciones nuevas
        tools=TOOLS,
        reasoning={"effort": "low", "summary": "auto"},
        store=True,
    )
else:
    print(f"⚠️  MAX_ITERATIONS ({MAX_ITERATIONS}) alcanzado — el Credibilómetro ha estado especialmente cruel hoy.")

comunicado_final = resp.output_text

## 5 · La traza completa

Lo que en la sesión llamamos *el material didáctico*: qué pensó, qué hizo y qué observó el agente
en cada paso. Aquí se ve la toma de decisiones: excusas descartadas por nota baja, re-intentos,
y la parada cuando junta las 5.

In [ ]:
print("=" * 78)
print("TRAZA DEL AGENTE DE EXCUSAS — razona → actúa → observa")
print("=" * 78)
for t in trace:
    print(f"\nSTEP {t['step']}")
    print(f"  reasoning:   {t['reasoning'][:400]}")
    for name, args, result in t["actions"]:
        print(f"  action:      {name}({json.dumps(args, ensure_ascii=False)})")
        print(f"  observation: {json.dumps(result, ensure_ascii=False)[:300]}")

certificadas = sorted({n for n, nota in mediciones if nota >= 7})
print(f"\n{'=' * 78}")
print(f"Mediciones totales: {len(mediciones)} · Excusas certificadas: {certificadas}")

In [ ]:
# ── Plot: el Credibilómetro™ en acción ─────────────────────────────────────
import matplotlib.pyplot as plt

if mediciones:
    etiquetas = [f"#{n}" for n, _ in mediciones]
    notas = [nota for _, nota in mediciones]
    colores = ["#2e7d32" if nota >= 7 else "#9e9e9e" for nota in notas]

    fig, ax = plt.subplots(figsize=(max(6, len(mediciones) * 0.7), 4))
    ax.bar(range(len(mediciones)), notas, color=colores)
    ax.axhline(7, color="#c62828", linestyle="--", linewidth=1.2, label="umbral de certificación (7)")
    ax.set_xticks(range(len(mediciones)), etiquetas)
    ax.set_ylim(0, 10.5)
    ax.set_xlabel("excusa medida (en orden de llamada)")
    ax.set_ylabel("credibilidad")
    ax.set_title("El Credibilómetro™ — verde: certificada · gris: los alumnos no se lo tragan")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No hay mediciones que plotear (¿ejecutaste el bucle?)")

## 6 · La salida final: el Comunicado Oficial 📜

In [ ]:
print("=" * 78)
print("COMUNICADO OFICIAL A LOS ALUMNOS DEL MÁSTER")
print("=" * 78)
print(comunicado_final)

## 7 · Lo que acabáis de ver (moraleja seria)

Quitadle el disfraz cómico y esto es un agente de manual:

1. **El modelo decidió el flujo**: qué excusas consultar, cuáles medir, cuáles descartar y cuándo parar.
   Nadie escribió ese orden en código — solo el *protocolo* en el system prompt.
2. **El entorno es no determinista** (el Credibilómetro aleatorio) y el agente **se adapta**: si una
   excusa suspende, busca otra. Ejecutad el notebook dos veces: la traza será distinta. Eso es
   exactamente lo que pasa con búsquedas RAG, APIs externas o usuarios reales.
3. **Dos frenos**: la parada natural (objetivo cumplido) y `MAX_ITERATIONS` (el seguro). Todo agente
   en producción necesita los dos.
4. **La traza lo es todo**: sin ella no puedes depurar, auditar, ni saber cuánto te costó. Con ella,
   hasta un desastre de Docker se convierte en material didáctico.

El agente real de la sesión (`run_agent_s12.py`) es esto mismo con tools de negocio:
`search_budgets` en vez de `buscar_excusa`, consenso de horas en vez de Credibilómetro.
La mecánica — Responses API, `previous_response_id`, dispatch, observaciones, trazas — es idéntica.